# Importing necessary libraries

In [ ]:
try:
  !pip install -q pdfplumber
  !pip install -q keybert
  import pdfplumber
  import keybert
except:
  "Packages already installed"
  import pdfplumber
  import keybert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.9 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT
import pdfplumber

#Skill Ontology

In [ ]:
GENERAL_SKILLS = [
    # Technical
    "python", "java", "c++", "sql", "machine learning", "data analysis",
    "deep learning", "cloud computing", "aws", "azure", "gcp",

    # Business & Management
    "project management", "business analysis", "strategic planning",
    "stakeholder management", "risk management", "operations management",

    # Marketing & Sales
    "digital marketing", "seo", "content marketing", "branding",
    "market research", "lead generation", "sales", "crm",

    # Finance
    "financial analysis", "accounting", "budgeting", "forecasting",
    "financial modeling", "investment analysis",

    # HR
    "recruitment", "talent acquisition", "employee engagement",
    "performance management", "payroll", "training and development",

    # Design & Creative
    "graphic design", "ui design", "ux design", "adobe photoshop",
    "illustrator", "figma", "video editing",

    # Healthcare & Education
    "patient care", "clinical research", "teaching", "curriculum development",

    # Soft Skills
    "communication", "leadership", "teamwork", "problem solving",
    "critical thinking", "time management", "adaptability"
]

# Reading the PDF

In [ ]:
def read_pdf(file_input):
    text = ""
    try:
        if file_input is None:
            return ""

        # Works for both file path and Gradio file object
        pdf = pdfplumber.open(file_input)

        with pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""

    except Exception as e:
        print("PDF ERROR:", e)
        return ""

    return text

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
kw_model = KeyBERT()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Skill Extraction

In [ ]:
import re

GENERIC_WORDS = {
    "required", "requirement", "responsibilities", "qualification",
    "qualifications", "preferred", "opportunity", "opportunities",
    "candidate", "candidates", "role", "position", "job", "work",
    "team", "company", "experience", "understanding", "knowledge"
}

def clean_phrase(phrase):
    words = phrase.split()
    if len(words) > 4:
        return False
    if any(word in GENERIC_WORDS for word in words):
        return False
    return True


def extract_skills(text):
    text = text.lower()
    detected_skills = set()

    # Match skills from ontology
    for skill in GENERAL_SKILLS:
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, text):
            detected_skills.add(skill)

    # Extract additional keywords using KeyBERT
    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=15
        )

        for keyword, _ in keywords:
            keyword = keyword.lower().strip()
            if clean_phrase(keyword):
                detected_skills.add(keyword)

    except Exception as e:
        print("KeyBERT Error:", e)

    return sorted(detected_skills)

# Critical skill extraction

In [ ]:
def extract_critical_skills(jd_text):
    jd_text = jd_text.lower()

    critical_keywords = [
        "must", "required", "requirement", "requirements",
        "mandatory", "essential", "minimum qualification",
        "need to have", "should have", "responsible for",
        "role requires", "requiring", "skills required"
    ]

    critical_skills = []
    lines = jd_text.split("\n")

    # Extract explicitly stated critical skills
    for line in lines:
        if any(keyword in line for keyword in critical_keywords):
            skills_in_line = extract_skills(line)
            for skill in skills_in_line:
                if skill not in GENERIC_WORDS:
                    critical_skills.append(skill)

    # Fallback: if no critical skills found, treat JD skills as critical
    if not critical_skills:
        critical_skills = extract_skills(jd_text)

    return list(set(critical_skills))

In [ ]:
def compute_semantic(resume_emb, jd_emb):
    return cosine_similarity([resume_emb], [jd_emb])[0][0]

In [ ]:
def compute_skill_score(resume_text, jd_skills):
    resume_skills = extract_skills(resume_text)

    if not jd_skills:
        return 0, [], resume_skills

    resume_embeddings = embed_model.encode(resume_skills)
    jd_embeddings = embed_model.encode(jd_skills)

    similarity_matrix = cosine_similarity(
        resume_embeddings,
        jd_embeddings
    )

    threshold = 0.70
    matches = []

    for i, jd_skill in enumerate(jd_skills):
        if similarity_matrix[:, i].max() >= threshold:
            matches.append(jd_skill)

    score = len(matches) / len(jd_skills)
    return score, matches, resume_skills

# Keyword Score

In [ ]:
def compute_keyword_score(resume, jd):
    tfidf = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=5000
    )
    matrix = tfidf.fit_transform([resume.lower(), jd.lower()])
    return cosine_similarity(matrix[0:1], matrix[1:2])[0][0]

# ATS Score Computation

In [ ]:
def compute_ats(resume, jd):
    if not resume or not jd:
        return {"Error": "Resume or Job Description is empty"}

    # Generate embeddings
    resume_emb = embed_model.encode(resume)
    jd_emb = embed_model.encode(jd)

    # Semantic Similarity
    semantic = compute_semantic(resume_emb, jd_emb)

    # Skill Extraction
    jd_skills = extract_skills(jd)
    critical_skills = extract_critical_skills(jd)

    skill_score, matched_skills, resume_skills = compute_skill_score(
        resume, jd_skills
    )

    # Critical Skill Matching
    if critical_skills:
        matched_critical = list(set(matched_skills) & set(critical_skills))
        critical_score = len(matched_critical) / len(critical_skills)
    else:
        matched_critical = []
        critical_score = 1

    # Keyword Similarity
    keyword_score = compute_keyword_score(resume, jd)

    # Weighted ATS Score
    ats = (
        0.40 * semantic +
        0.25 * skill_score +
        0.20 * critical_score +
        0.10 * keyword_score +
        0.05
    ) * 100

    ats = max(0, min(100, ats))

    # Summary scores only
    summary = {
        "ATS Score": round(ats, 2),
        "Semantic Score": round(semantic, 3),
        "Skill Match Score": round(skill_score, 3),
        "Critical Skill Score": round(critical_score, 3),
        "Keyword Match Score": round(keyword_score, 3),
    }

    return {**summary}

# Benchmarking against CrossEncoder

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/stsb-roberta-base"
)

import numpy as np

def evaluate_with_cross_encoder(resume, jd):
    """
    Evaluate similarity using a semantic similarity Cross-Encoder.
    Returns scores normalized to a 0–100 scale.
    """
    # Clean and truncate inputs
    resume = " ".join(resume.split())[:1500]
    jd = " ".join(jd.split())[:500]

    # Predict similarity score
    raw_score = cross_encoder.predict([(resume, jd)])[0]

    # STSB models typically output scores in the range 0–5
    normalized_score = max(0, min(5, raw_score)) / 5
    percentage_score = normalized_score * 100

    return {
        "Semantic Score": round(normalized_score, 3),
        "Skill Match Score": None,
        "Keyword Match Score": None,
        "Critical Skill Score": None,
        "ATS Score": round(percentage_score, 2)
    }


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
def compare_models(resume, jd):
    custom = compute_ats(resume, jd)
    cross = evaluate_with_cross_encoder(resume, jd)

    comparison = {
        "Custom ATS": {
            "ATS Score (%)": custom["ATS Score"],
            "Semantic Score": custom["Semantic Score"],
            "Skill Match Score": custom["Skill Match Score"],
            "Critical Skill Score": custom["Critical Skill Score"],
            "Keyword Match Score": custom["Keyword Match Score"],
        },
        "Cross-Encoder Benchmark": {
            "Semantic Logit Score": cross["Semantic Score"],
            "Skill Match Score": "N/A",
            "Critical Skill Score": "N/A",
            "Keyword Match Score": "N/A",
            "ATS Score (%)": "N/A",
        }
    }

    return pd.DataFrame(comparison).T

In [ ]:
import pandas as pd

resume_path = "Harman_Resume.pdf"
resume_text = read_pdf(resume_path)


# Test Case 1: Aligned Role
jd_aligned = """
Machine Learning Engineer role requiring Python, SQL,
Machine Learning, Deep Learning, Data Analysis, TensorFlow,
PyTorch, and Scikit-learn.
"""

# Test Case 2: Partially Aligned Role
jd_partial = """
Data Analyst role requiring SQL, Excel, Data Visualization,
Statistics, Tableau, and Power BI.
"""

# Test Case 3: Unrelated Role
jd_unrelated = """
Marketing Manager role requiring SEO, Branding,
Digital Marketing, Social Media Management, and Content Strategy.
"""

test_cases = [
    ("Aligned Profile – Machine Learning Engineer", jd_aligned),
    ("Partially Aligned Profile – Data Analyst", jd_partial),
    ("Unrelated Profile – Marketing Manager", jd_unrelated),
]


results = []

for name, jd in test_cases:
    custom = compute_ats(resume_text, jd)
    cross = evaluate_with_cross_encoder(resume_text, jd)

    results.append({
        "Test Case": name,
        "Custom ATS Score (%)": custom["ATS Score"],
        "Custom Semantic Score": custom["Semantic Score"],
        "Cross-Encoder Logit": cross["Semantic Score"]
    })

results_df = pd.DataFrame(results)
results_df

,Test Case,Custom ATS Score (%),Custom Semantic Score,Cross-Encoder Logit
0,Aligned Profile – Machine Learning Engineer,43.87,0.565,0.102
1,Partially Aligned Profile – Data Analyst,31.54,0.465,0.079
2,Unrelated Profile – Marketing Manager,12.46,0.185,0.075


# Analyzing for Enhancement

In [ ]:
def analyze_resume_jd(resume_text, jd_text):

    jd_skills = extract_skills(jd_text)
    resume_skills = extract_skills(resume_text)

    matched_skills = list(set(jd_skills) & set(resume_skills))
    missing_skills = list(set(jd_skills) - set(resume_skills))

    match_ratio = len(matched_skills) / len(jd_skills) if jd_skills else 0

    if match_ratio > 0.7:
        level = "High"
    elif match_ratio > 0.4:
        level = "Medium"
    else:
        level = "Low"

    suggestions = []

    tech_skills = [s for s in missing_skills if s not in ["communication", "leadership", "teamwork"]]
    soft_skills = [s for s in missing_skills if s in ["communication", "leadership", "teamwork"]]

    if tech_skills:
        suggestions.append(
            f"Build projects using {', '.join(tech_skills[:3])} to demonstrate practical experience"
        )
        suggestions.append(
            f"Add {', '.join(tech_skills[:3])} explicitly in your resume with project-based examples"
        )

    if soft_skills:
        suggestions.append(
            f"Demonstrate {', '.join(soft_skills)} through teamwork, presentations, or leadership roles"
        )

    if matched_skills:
        suggestions.append("Strengthen existing skills with measurable achievements and real-world impact")

    if 0.3 <= match_ratio <= 0.7:
        suggestions.append("Bridge the gap between your current skills and job requirements with targeted learning")

    if match_ratio < 0.3:
        suggestions.append("Align your resume more closely with the job description and required keywords")

    suggestions = list(dict.fromkeys(suggestions))[:5]

    return f"""
Match Level: {level}

Matched Skills:
{', '.join(matched_skills) if matched_skills else 'None'}

Missing Skills:
{', '.join(missing_skills) if missing_skills else 'None'}

Suggestions:
- {'\n- '.join(suggestions)}
"""

In [ ]:
def process_inputs(resume_file, resume_text,jd_text):
    try:
        resume = read_pdf(resume_file) if resume_file else resume_text
        jd = jd_text

        if not resume or not jd:
            return "Please provide both Resume and Job Description.", ""

        result = compute_ats(resume, jd)

        suggestions = analyze_resume_jd(resume, jd)

        ats_output = f"""
📊 ATS Score: {result['ATS Score']}%

📈 Breakdown:
• Semantic Score: {result['Semantic Score']}
• Skill Match: {result['Skill Match Score']}
• Critical Skills: {result['Critical Skill Score']}
• Keyword Match: {result['Keyword Match Score']}
"""

        return ats_output.strip(), suggestions.strip()

    except Exception as e:
        return f" Error: {str(e)}", "⚠️ Something went wrong."

In [ ]:
import gradio as gr

def reset_all():
    return None, "","", "", ""

with gr.Blocks() as demo:

    gr.Markdown("# 📄 ATS Resume Checker")

    with gr.Row():
        with gr.Column():
            resume_file = gr.File(label="Upload Resume (PDF)", file_types=[".pdf"])
            resume_text = gr.Textbox(lines=10, label="Or paste Resume Text")

        with gr.Column():
            jd_text = gr.Textbox(label="Paste Job Description")

    submit_btn = gr.Button("🚀 Analyze Resume")
    reset_btn = gr.Button("🔄 Reset")

    output = gr.Textbox(label="ATS Result", lines=8)
    suggestion_box = gr.Textbox(label="Suggestions", lines=10)

    submit_btn.click(
        fn=process_inputs,
        inputs=[resume_file, resume_text, jd_text],
        outputs=[output, suggestion_box]
    )

    reset_btn.click(
        fn=reset_all,
        inputs=[],
        outputs=[resume_file, resume_text, jd_text, output, suggestion_box]
    )
demo.queue()
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e65fd91fc6a8c5f742.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
